# LeetCode 1247 — Minimum Swaps to Make Strings Equal bằng Python

Link đề: https://leetcode.com/problems/minimum-swaps-to-make-strings-equal/ · Độ khó: **Medium** · Chủ đề: Math, String, Greedy

Skeleton LeetCode:

```python
class Solution:
    def minimumSwap(self, s1: str, s2: str) -> int:
        ...
```

Ghi chú: phép đổi chỗ (swap) chỉ được thực hiện giữa **hai chuỗi khác nhau**: `s1[i] ↔ s2[j]` (với `i`, `j` bất kỳ, có thể khác nhau). Không được đổi hai ký tự trong cùng một chuỗi.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Mô hình hoá bài toán "số bước ít nhất" thành **BFS trên không gian trạng thái** để có lời giải vét cạn làm chuẩn kiểm thử.
2. Nhận ra chỉ các vị trí **lệch** mới quan trọng, và chia chúng thành hai loại: `xy` (`s1[i]='x', s2[i]='y'`) và `yx` (ngược lại).
3. Hiểu hai "nước đi" then chốt: một cặp **cùng loại** (`xy`+`xy` hoặc `yx`+`yx`) sửa bằng **1** swap; một cặp **khác loại** (`xy`+`yx`) cần **2** swap.
4. Chứng minh điều kiện vô nghiệm bằng **tính chẵn lẻ** (parity): tổng số `x` phải chẵn.
5. Viết công thức `O(n)` và tự dựng được **dãy swap cụ thể** để kiểm chứng.

## 2. Đề bài & ví dụ

Cho hai chuỗi `s1`, `s2` **cùng độ dài**, chỉ gồm chữ `'x'` và `'y'`. Nhiệm vụ: làm hai chuỗi bằng nhau. Được phép đổi chỗ **hai ký tự thuộc hai chuỗi khác nhau**, tức là swap `s1[i]` với `s2[j]`.

Trả về **số swap ít nhất** để `s1 == s2`, hoặc `-1` nếu không thể.

**Ví dụ 1:** `s1 = "xx", s2 = "yy"` → `1` (swap `s1[0]` với `s2[1]`: `s1 = "yx"`, `s2 = "yx"`).

**Ví dụ 2:** `s1 = "xy", s2 = "yx"` → `2` (swap `s1[0]↔s2[0]`: `"yy"`, `"xx"`; rồi `s1[0]↔s2[1]`: `"xy"`, `"xy"`). Không được swap `s1[0]` với `s1[1]`.

**Ví dụ 3:** `s1 = "xx", s2 = "xy"` → `-1`.

**Ràng buộc:** `1 <= s1.length, s2.length <= 1000`; `s1.length == s2.length`; chỉ gồm `'x'` hoặc `'y'`.

In [1]:
import itertools
import random
import time
from collections import deque, Counter
from typing import List

examples = [("xx", "yy", 1), ("xy", "yx", 2), ("xx", "xy", -1)]
for s1, s2, want in examples:
    print(f"s1={s1!r}, s2={s2!r} -> {want}")

s1='xx', s2='yy' -> 1
s1='xy', s2='yx' -> 2
s1='xx', s2='xy' -> -1


## 3. Kiến thức nền

### 3.1 Duyệt song song hai chuỗi với `zip`

`zip(s1, s2)` cho từng cặp `(s1[i], s2[i])`. Đây là cách gọn để so sánh hai chuỗi cùng độ dài theo vị trí.

In [2]:
s1, s2 = "xxyyxyxyxx", "xyyxyxxxyx"
for i, (a, b) in enumerate(zip(s1, s2)):
    tag = "khớp" if a == b else ("xy" if a == "x" else "yx")
    print(i, a, b, tag)

0 x x khớp
1 x y xy
2 y y khớp
3 y x yx
4 x y xy
5 y x yx
6 x x khớp
7 y x yx
8 x y xy
9 x x khớp


### 3.2 Đếm bằng `sum` trên biểu thức boolean và `Counter`

Trong Python `True == 1`, nên `sum(a == 'x' and b == 'y' for a, b in zip(s1, s2))` đếm số vị trí loại `xy`. `Counter(zip(s1, s2))` đếm mọi loại cặp một lần.

In [3]:
xy = sum(a == "x" and b == "y" for a, b in zip(s1, s2))
yx = sum(a == "y" and b == "x" for a, b in zip(s1, s2))
print("xy =", xy, "| yx =", yx)
print(Counter(zip(s1, s2)))

xy = 3 | yx = 3
Counter({('x', 'x'): 3, ('x', 'y'): 3, ('y', 'x'): 3, ('y', 'y'): 1})


### 3.3 Chuỗi bất biến → dùng `list` để swap

Chuỗi Python là **immutable** (không sửa được tại chỗ). Muốn mô phỏng swap, chuyển sang `list`, đổi chỗ bằng gán bộ (tuple assignment), rồi `''.join` lại.

In [4]:
a, b = list("xy"), list("yx")
a[0], b[0] = b[0], a[0]          # swap s1[0] <-> s2[0]
print("".join(a), "".join(b))
a[0], b[1] = b[1], a[0]          # swap s1[0] <-> s2[1]
print("".join(a), "".join(b))

yy xx
xy xy


### 3.4 BFS tìm số bước ít nhất

BFS (Breadth-First Search) trên đồ thị không trọng số tìm **đường ngắn nhất** (ít bước nhất) từ trạng thái đầu tới trạng thái đích. Mỗi trạng thái ở đây là cặp `(s1, s2)`; mỗi cạnh là một swap hợp lệ.

In [5]:
def bfs_min_steps(start, is_goal, neighbors):
    dist = {start: 0}
    q = deque([start])
    while q:
        u = q.popleft()
        if is_goal(u):
            return dist[u]
        for v in neighbors(u):
            if v not in dist:
                dist[v] = dist[u] + 1
                q.append(v)
    return -1


# ví dụ đồ chơi: từ 1 tới 10, mỗi bước +1 hoặc *2
print(bfs_min_steps(1, lambda x: x == 10, lambda x: [x + 1, x * 2] if x < 20 else []))

4


## 4. Ý tưởng nền tảng / trực giác

**Vị trí khớp không cần đụng vào.** Chỉ các vị trí lệch mới quan trọng. Có hai loại lệch:

- `xy`: `s1[i] = 'x'`, `s2[i] = 'y'`
- `yx`: `s1[i] = 'y'`, `s2[i] = 'x'`

**Tính tay hai tình huống nhỏ:**

1. Hai vị trí cùng loại `xy` (ví dụ 1: `"xx"/"yy"`): swap `s1[0]` với `s2[1]` → `s1 = "yx"`, `s2 = "yx"`. **1 swap sửa 2 vị trí.**
2. Một `xy` và một `yx` (ví dụ 2: `"xy"/"yx"`): swap chéo không giúp ngay; swap `s1[0]↔s2[0]` biến thành hai vị trí **cùng loại** (`"yy"/"xx"` — hai `yx`), rồi 1 swap nữa. **2 swap sửa 2 vị trí.**

Vậy chiến lược: ghép cặp `xy` với `xy`, `yx` với `yx` (1 swap/cặp); nếu dư mỗi loại một cái thì tốn thêm 2 swap. Nếu dư **chỉ một** cái (tổng lệch lẻ) thì vô nghiệm.

In [6]:
s1, s2 = "xxyyxyxyxx", "xyyxyxxxyx"
xy = sum(a == "x" and b == "y" for a, b in zip(s1, s2))
yx = sum(a == "y" and b == "x" for a, b in zip(s1, s2))
print(f"xy={xy}, yx={yx}")
print(f"cặp xy: {xy // 2}, cặp yx: {yx // 2}, dư xy: {xy % 2}, dư yx: {yx % 2}")
print("dự đoán:", -1 if (xy + yx) % 2 else xy // 2 + yx // 2 + 2 * (xy % 2))

xy=3, yx=3
cặp xy: 1, cặp yx: 1, dư xy: 1, dư yx: 1
dự đoán: 4


## 5. Cách 1 — Brute Force / baseline

BFS trên mọi trạng thái `(s1, s2)`. Từ mỗi trạng thái, thử mọi swap `s1[i] ↔ s2[j]` (`n²` cách). Dừng khi `s1 == s2`. Vì BFS duyệt theo số bước tăng dần, lần đầu gặp đích chính là số swap ít nhất. Nếu duyệt hết mà không gặp → `-1`.

In [7]:
class SolutionBFS:
    def minimumSwap(self, s1, s2):
        n = len(s1)

        def neighbors(state):
            a, b = state
            out = []
            for i in range(n):
                for j in range(n):
                    if a[i] != b[j]:                 # swap hai ký tự giống nhau là vô ích
                        la, lb = list(a), list(b)
                        la[i], lb[j] = lb[j], la[i]
                        out.append(("".join(la), "".join(lb)))
            return out

        return bfs_min_steps((s1, s2), lambda st: st[0] == st[1], neighbors)


for s1, s2, want in examples:
    print(SolutionBFS().minimumSwap(s1, s2), want)

1 1
2 2
-1 -1


**Độ phức tạp:** số trạng thái có thể tới `C(2n, số x)` — hàm mũ theo `n`; mỗi trạng thái sinh `n²` láng giềng. Chỉ dùng được với `n ≤ 5` hoặc 6. **Nút thắt:** ta coi mọi vị trí như nhau, trong khi chỉ **số lượng** vị trí lệch từng loại mới quan trọng.

## 6. Bước đột phá

**Insight 1 — Chỉ cần hai con số `xy` và `yx`.** Vị trí khớp không bao giờ cần swap (swap chúng chỉ phá thêm). Hai vị trí cùng loại hoán đổi vai trò cho nhau được, nên trạng thái thực chất chỉ là cặp `(xy, yx)`.

**Insight 2 — Mỗi swap sửa được tối đa 2 vị trí lệch**, và sửa đủ 2 **chỉ khi** hai vị trí đó cùng loại. Với hai vị trí khác loại, cần 2 swap.

**Insight 3 — Điều kiện có nghiệm (parity):** ở trạng thái đích, mỗi vị trí có hai ký tự giống nhau → tổng số `'x'` trong `s1 + s2` chẵn. Swap không đổi tổng số `'x'`. Vị trí khớp đóng góp 0 hoặc 2 chữ `x`; vị trí lệch đóng góp đúng 1. Vậy tổng số `x` ≡ `xy + yx` (mod 2). **Nếu `xy + yx` lẻ thì `-1`.**

**Công thức:** khi `xy + yx` chẵn thì `xy`, `yx` cùng chẵn hoặc cùng lẻ:

- cùng chẵn: `xy/2 + yx/2`;
- cùng lẻ: `(xy−1)/2 + (yx−1)/2 + 2`.

Gộp lại: `xy // 2 + yx // 2 + 2 * (xy % 2)`, hoặc gọn hơn `(xy + 1) // 2 + (yx + 1) // 2`.

In [8]:
for xy_, yx_ in [(2, 0), (1, 1), (3, 1), (4, 2), (3, 3), (0, 0)]:
    f1 = xy_ // 2 + yx_ // 2 + 2 * (xy_ % 2)
    f2 = (xy_ + 1) // 2 + (yx_ + 1) // 2
    print(f"xy={xy_}, yx={yx_}: công thức 1 = {f1}, công thức 2 = {f2}")

xy=2, yx=0: công thức 1 = 1, công thức 2 = 1
xy=1, yx=1: công thức 1 = 2, công thức 2 = 2
xy=3, yx=1: công thức 1 = 3, công thức 2 = 3
xy=4, yx=2: công thức 1 = 3, công thức 2 = 3
xy=3, yx=3: công thức 1 = 4, công thức 2 = 4
xy=0, yx=0: công thức 1 = 0, công thức 2 = 0


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Đếm + công thức

In [9]:
class SolutionCount:
    def minimumSwap(self, s1, s2):
        xy = yx = 0
        for a, b in zip(s1, s2):
            if a == "x" and b == "y":
                xy += 1
            elif a == "y" and b == "x":
                yx += 1
        if (xy + yx) % 2:
            return -1
        return xy // 2 + yx // 2 + 2 * (xy % 2)


for s1, s2, want in examples:
    print(SolutionCount().minimumSwap(s1, s2), want)

1 1
2 2
-1 -1


**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ.

### 7.2 Cách 3 — Tham lam có dựng dãy swap cụ thể

Để chắc chắn công thức không "ảo", ta dựng **danh sách swap** `(i, j)` (nghĩa là swap `s1[i]↔s2[j]`) rồi áp dụng thật và kiểm tra `s1 == s2`.

- Ghép từng cặp vị trí `xy` `(p, q)`: swap `s1[p] ↔ s2[q]`.
- Ghép từng cặp vị trí `yx` `(p, q)`: swap `s1[p] ↔ s2[q]`.
- Nếu dư một `xy` tại `p` và một `yx` tại `q`: swap `s1[p] ↔ s2[p]` (biến `p` thành loại `yx`), rồi swap `s1[p] ↔ s2[q]`.

In [10]:
def build_swaps(s1, s2):
    XY = [i for i, (a, b) in enumerate(zip(s1, s2)) if a == "x" and b == "y"]
    YX = [i for i, (a, b) in enumerate(zip(s1, s2)) if a == "y" and b == "x"]
    if (len(XY) + len(YX)) % 2:
        return None
    swaps = []
    for k in range(0, len(XY) - 1, 2):
        swaps.append((XY[k], XY[k + 1]))
    for k in range(0, len(YX) - 1, 2):
        swaps.append((YX[k], YX[k + 1]))
    if len(XY) % 2:
        p, q = XY[-1], YX[-1]
        swaps.append((p, p))
        swaps.append((p, q))
    return swaps


def apply_swaps(s1, s2, swaps):
    a, b = list(s1), list(s2)
    for i, j in swaps:
        a[i], b[j] = b[j], a[i]
    return "".join(a), "".join(b)


class SolutionConstruct:
    def minimumSwap(self, s1, s2):
        swaps = build_swaps(s1, s2)
        if swaps is None:
            return -1
        a, b = apply_swaps(s1, s2, swaps)
        assert a == b, "dãy swap dựng sai"
        return len(swaps)


for s1, s2, want in examples:
    print(SolutionConstruct().minimumSwap(s1, s2), want, build_swaps(s1, s2))

1 1 [(0, 1)]
2 2 [(0, 0), (0, 1)]
-1 -1 None


**Độ phức tạp:** `O(n)` thời gian và bộ nhớ (lưu danh sách vị trí và swap). Lợi ích: có "bằng chứng" (witness) rằng số swap trả về là **đạt được**.

### 7.3 Cách 4 — Công thức gọn bằng `Counter` và parity tổng số `x`

Một cách viết khác: kiểm tra vô nghiệm bằng `(s1 + s2).count('x') % 2`, đếm loại cặp bằng `Counter(zip(...))`, và dùng `(xy + 1) // 2 + (yx + 1) // 2`.

In [11]:
class SolutionCompact:
    def minimumSwap(self, s1, s2):
        if (s1 + s2).count("x") % 2:
            return -1
        c = Counter(zip(s1, s2))
        xy, yx = c[("x", "y")], c[("y", "x")]
        return (xy + 1) // 2 + (yx + 1) // 2


for s1, s2, want in examples:
    print(SolutionCompact().minimumSwap(s1, s2), want)

1 1
2 2
-1 -1


Vì sao `(xy + 1) // 2 + (yx + 1) // 2` đúng: khi cùng chẵn thì bằng `xy/2 + yx/2`; khi cùng lẻ thì bằng `(xy+1)/2 + (yx+1)/2 = (xy−1)/2 + (yx−1)/2 + 2`. Khớp với công thức ở mục 6.

### 7.4 Cách 5 — BFS trên trạng thái thu gọn `(xy, yx)`

Áp dụng Insight 1: trạng thái chỉ là `(xy, yx)`. Các nước đi có thể: swap hai vị trí cùng loại `xy` → `(xy−2, yx)`; cùng loại `yx` → `(xy, yx−2)`; swap dọc tại một vị trí lệch (`s1[p]↔s2[p]`) đổi loại: `(xy−1, yx+1)` hoặc `(xy+1, yx−1)`. (Các swap khác — giữa vị trí lệch và vị trí khớp... — không có lợi; ta chỉ dùng BFS này để minh hoạ, đã kiểm chứng bằng Cách 1 ở mục 12.)

In [12]:
class SolutionStateBFS:
    def minimumSwap(self, s1, s2):
        xy = sum(a == "x" and b == "y" for a, b in zip(s1, s2))
        yx = sum(a == "y" and b == "x" for a, b in zip(s1, s2))

        def nb(st):
            p, q = st
            out = []
            if p >= 2:
                out.append((p - 2, q))
            if q >= 2:
                out.append((p, q - 2))
            if p >= 1:
                out.append((p - 1, q + 1))
            if q >= 1:
                out.append((p + 1, q - 1))
            return out

        return bfs_min_steps((xy, yx), lambda st: st == (0, 0), nb)


for s1, s2, want in examples:
    print(SolutionStateBFS().minimumSwap(s1, s2), want)

1 1
2 2
-1 -1


**Độ phức tạp:** số trạng thái `O(n)` (vì `xy + yx` không đổi), mỗi trạng thái ≤ 4 láng giềng → `O(n)`.

## 8. TRACE từng bước

Trace Cách 2 trên `s1 = "xxyyxyxyxx"`, `s2 = "xyyxyxxxyx"`:

| i | s1[i] | s2[i] | loại | xy | yx |
|---|---|---|---|---|---|
| 0 | x | x | khớp | 0 | 0 |
| 1 | x | y | xy | 1 | 0 |
| 2 | y | y | khớp | 1 | 0 |
| 3 | y | x | yx | 1 | 1 |
| 4 | x | y | xy | 2 | 1 |
| 5 | y | x | yx | 2 | 2 |
| 6 | x | x | khớp | 2 | 2 |
| 7 | y | x | yx | 2 | 3 |
| 8 | x | y | xy | 3 | 3 |
| 9 | x | x | khớp | 3 | 3 |

`xy + yx = 6` chẵn → `3 // 2 + 3 // 2 + 2 · 1 = 1 + 1 + 2 = 4`.

In [13]:
def trace_count(s1, s2):
    xy = yx = 0
    for i, (a, b) in enumerate(zip(s1, s2)):
        if a == "x" and b == "y":
            xy += 1
            kind = "xy"
        elif a == "y" and b == "x":
            yx += 1
            kind = "yx"
        else:
            kind = "khớp"
        print(f"i={i}: {a}{b} -> {kind:4} | xy={xy}, yx={yx}")
    if (xy + yx) % 2:
        print("tổng lệch lẻ -> -1")
        return -1
    ans = xy // 2 + yx // 2 + 2 * (xy % 2)
    print(f"đáp án = {xy}//2 + {yx}//2 + 2*({xy}%2) = {ans}")
    return ans


trace_count("xxyyxyxyxx", "xyyxyxxxyx")

i=0: xx -> khớp | xy=0, yx=0
i=1: xy -> xy   | xy=1, yx=0
i=2: yy -> khớp | xy=1, yx=0
i=3: yx -> yx   | xy=1, yx=1
i=4: xy -> xy   | xy=2, yx=1
i=5: yx -> yx   | xy=2, yx=2
i=6: xx -> khớp | xy=2, yx=2
i=7: yx -> yx   | xy=2, yx=3
i=8: xy -> xy   | xy=3, yx=3
i=9: xx -> khớp | xy=3, yx=3
đáp án = 3//2 + 3//2 + 2*(3%2) = 4


4

Trace dãy swap dựng bởi Cách 3 trên cùng input, in hai chuỗi sau mỗi swap:

In [14]:
def trace_swaps(s1, s2):
    swaps = build_swaps(s1, s2)
    a, b = list(s1), list(s2)
    print("đầu   :", "".join(a), "".join(b))
    for i, j in swaps:
        a[i], b[j] = b[j], a[i]
        print(f"s1[{i}]<->s2[{j}]:", "".join(a), "".join(b))
    print("bằng nhau:", a == b, "| số swap:", len(swaps))


trace_swaps("xxyyxyxyxx", "xyyxyxxxyx")

đầu   : xxyyxyxyxx xyyxyxxxyx
s1[1]<->s2[4]: xyyyxyxyxx xyyxxxxxyx
s1[3]<->s2[5]: xyyxxyxyxx xyyxxyxxyx
s1[8]<->s2[8]: xyyxxyxyyx xyyxxyxxxx
s1[8]<->s2[7]: xyyxxyxyxx xyyxxyxyxx
bằng nhau: True | số swap: 4


## 9. Code nộp LeetCode

In [15]:
class Solution:
    def minimumSwap(self, s1: str, s2: str) -> int:
        xy = yx = 0
        for a, b in zip(s1, s2):
            if a != b:
                if a == "x":
                    xy += 1
                else:
                    yx += 1
        if (xy + yx) % 2:
            return -1
        return xy // 2 + yx // 2 + 2 * (xy % 2)


for s1, s2, want in examples:
    assert Solution().minimumSwap(s1, s2) == want
print("Solution OK trên ví dụ đề")

Solution OK trên ví dụ đề


## 10. Vì sao đúng

**Vô nghiệm ⇔ `xy + yx` lẻ.** Đã chứng minh ở Insight 3 (một chiều: lẻ thì vô nghiệm); chiều ngược lại do Cách 3 dựng được lời giải khi chẵn.

**Cận dưới.** Xét đại lượng `Φ = (xy + 1) // 2 + (yx + 1) // 2` (chính là công thức). Ta chứng minh mọi swap giảm `Φ` **nhiều nhất 1**:

- Một swap `s1[i] ↔ s2[j]` chỉ thay đổi tình trạng của vị trí `i` và vị trí `j` (cột `i` mất `s1[i]`, nhận `s2[j]`; cột `j` mất `s2[j]`, nhận `s1[i]`).
- Nếu `i = j`: chỉ đổi loại cột `i` (`xy ↔ yx`) hoặc không đổi gì → `(xy, yx)` thành `(xy ∓ 1, yx ± 1)`; `Φ` giảm tối đa 1 (kiểm tra bằng máy ở ô dưới).
- Nếu `i ≠ j`: số vị trí lệch thay đổi tối đa 2, và chỉ giảm đúng 2 khi hai cột cùng loại → `xy` (hoặc `yx`) giảm 2 → `Φ` giảm đúng 1. Các trường hợp còn lại `Φ` giảm ≤ 1.

Đích có `Φ = 0`, ban đầu `Φ = công thức`, nên cần ít nhất `Φ` swap. **Cận trên:** Cách 3 dựng đúng `Φ` swap. Vậy công thức là tối ưu. ∎

Ô dưới kiểm tra mệnh đề "mỗi swap giảm `Φ` tối đa 1" bằng cách thử **mọi** swap trên mọi cặp chuỗi độ dài ≤ 4.

In [16]:
def phi(s1, s2):
    xy = sum(a == "x" and b == "y" for a, b in zip(s1, s2))
    yx = sum(a == "y" and b == "x" for a, b in zip(s1, s2))
    return (xy + 1) // 2 + (yx + 1) // 2


ok = True
for n in range(1, 5):
    for t1 in itertools.product("xy", repeat=n):
        for t2 in itertools.product("xy", repeat=n):
            before = phi(t1, t2)
            for i in range(n):
                for j in range(n):
                    a, b = list(t1), list(t2)
                    a[i], b[j] = b[j], a[i]
                    if phi(a, b) < before - 1:
                        ok = False
print("mỗi swap giảm Φ tối đa 1:", ok)

mỗi swap giảm Φ tối đa 1: True


## 11. Time / Space Complexity

Đặt `n = len(s1)`.

| Cách | Time | Space |
|---|---|---|
| 1. BFS trạng thái đầy đủ | hàm mũ (`~C(2n, n) · n²`) | hàm mũ |
| 2. Đếm + công thức | `O(n)` | `O(1)` |
| 3. Dựng dãy swap | `O(n)` | `O(n)` |
| 4. `Counter` + parity | `O(n)` | `O(1)` (Counter ≤ 4 khoá; `s1 + s2` tạo chuỗi `O(n)`) |
| 5. BFS trạng thái `(xy, yx)` | `O(n)` | `O(n)` |

## 12. Bộ test

### 12.1 Ví dụ đề và edge cases

In [17]:
ALL = [SolutionCount, SolutionConstruct, SolutionCompact, SolutionStateBFS, Solution]


def check(s1, s2, want=None, with_bfs=True):
    outs = [cls().minimumSwap(s1, s2) for cls in ALL]
    if with_bfs:
        outs.append(SolutionBFS().minimumSwap(s1, s2))
    assert len(set(outs)) == 1, (s1, s2, outs)
    if want is not None:
        assert outs[0] == want, (s1, s2, outs[0], want)
    return outs[0]


for s1, s2, want in examples:
    check(s1, s2, want)
check("x", "x", 0)
check("x", "y", -1)
check("xyxy", "xyxy", 0)                 # đã bằng nhau
check("xxxx", "yyyy", 2)                 # 4 xy -> 2 swap
check("xy", "xy", 0)
check("xxyy", "yyxx", 2)                 # 2 xy + 2 yx -> 1 + 1
check("xxy", "yyx", -1)                  # xy=2, yx=1: tổng 3 lẻ -> -1 (bẫy!)
check("xyx", "yxx", 2)                   # xy=1, yx=1 -> 2
check("xxxy", "yyyx", 3)                 # xy=3, yx=1 -> 1 + 0 + 2
print("edge cases OK")

edge cases OK


Chú ý bẫy `"xxy"/"yyx"`: nhìn qua tưởng "3 vị trí lệch, cứ swap là xong", nhưng tổng lệch **lẻ** nên vô nghiệm. Còn `"xxxy"/"yyyx"` (`xy = 3`, `yx = 1`, tổng chẵn) cần `3` swap: một swap cho cặp `xy` + hai swap cho cặp dư khác loại.

### 12.2 Vét cạn toàn bộ chuỗi độ dài ≤ 4 + ngẫu nhiên có seed

In [18]:
cnt = 0
for n in range(1, 5):
    for t1 in itertools.product("xy", repeat=n):
        for t2 in itertools.product("xy", repeat=n):
            check("".join(t1), "".join(t2))
            cnt += 1
print("vét cạn", cnt, "cặp chuỗi (n <= 4) OK")

rng = random.Random(1247)
for _ in range(300):
    n = rng.randint(1, 5)
    s1 = "".join(rng.choice("xy") for _ in range(n))
    s2 = "".join(rng.choice("xy") for _ in range(n))
    check(s1, s2)
for _ in range(500):
    n = rng.randint(1, 200)
    s1 = "".join(rng.choice("xy") for _ in range(n))
    s2 = "".join(rng.choice("xy") for _ in range(n))
    check(s1, s2, with_bfs=False)
print("800 test ngẫu nhiên OK")

vét cạn 340 cặp chuỗi (n <= 4) OK


800 test ngẫu nhiên OK


### 12.3 Test giới hạn `n = 1000`

In [19]:
rng = random.Random(1)
s1 = "".join(rng.choice("xy") for _ in range(1000))
s2 = "".join(rng.choice("xy") for _ in range(1000))
if (s1 + s2).count("x") % 2:
    s2 = ("y" if s2[0] == "x" else "x") + s2[1:]   # chỉnh để có nghiệm
t = time.perf_counter()
r = check(s1, s2, with_bfs=False)
print("n = 1000 ->", r, f"({(time.perf_counter() - t) * 1000:.1f} ms cho 5 cách)")
assert check("x" * 1000, "y" * 1000, with_bfs=False) == 500

n = 1000 -> 249 (66.3 ms cho 5 cách)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Không phân loại vị trí lệch, lấy `số lệch / 2`

In [20]:
def wrong_half(s1, s2):
    diff = sum(a != b for a, b in zip(s1, s2))
    return -1 if diff % 2 else diff // 2


print("sai :", wrong_half("xy", "yx"), "| đúng:", Solution().minimumSwap("xy", "yx"))

sai : 1 | đúng: 2


**Vì sao sai:** `"xy"/"yx"` có 2 vị trí lệch nhưng **khác loại**, cần 2 swap chứ không phải 1. **Sửa:** đếm riêng `xy` và `yx`; cặp khác loại tốn 2.

### Lỗi 2 — Kiểm tra vô nghiệm sai: đòi `xy` và `yx` cùng chẵn

In [21]:
def wrong_parity(s1, s2):
    xy = sum(a == "x" and b == "y" for a, b in zip(s1, s2))
    yx = sum(a == "y" and b == "x" for a, b in zip(s1, s2))
    if xy % 2 or yx % 2:
        return -1                      # SAI: cùng lẻ vẫn giải được
    return xy // 2 + yx // 2


print("sai :", wrong_parity("xy", "yx"), "| đúng:", Solution().minimumSwap("xy", "yx"))

sai : -1 | đúng: 2


**Vì sao sai:** ví dụ 2 của đề có `xy = yx = 1` (cùng lẻ) và đáp án là `2`. Điều kiện đúng là **tổng** `xy + yx` chẵn. **Sửa:** `if (xy + yx) % 2: return -1`, và cộng thêm `2` khi cả hai lẻ.

### Lỗi 3 — Quên cộng `2` cho cặp dư khác loại

In [22]:
def wrong_forget_two(s1, s2):
    xy = sum(a == "x" and b == "y" for a, b in zip(s1, s2))
    yx = sum(a == "y" and b == "x" for a, b in zip(s1, s2))
    if (xy + yx) % 2:
        return -1
    return xy // 2 + yx // 2           # SAI: thiếu 2 * (xy % 2)


print("sai :", wrong_forget_two("xxy", "yxx"), "| đúng:", Solution().minimumSwap("xxy", "yxx"))

sai : 0 | đúng: 2


**Vì sao sai:** `"xxy"/"yxx"` có `xy = 1`, `yx = 1` → `1 // 2 + 1 // 2 = 0`, trong khi hai chuỗi chưa bằng nhau! **Sửa:** cộng `2 * (xy % 2)`.

### Lỗi 4 — Swap trong cùng một chuỗi (hiểu sai đề)

In [23]:
def wrong_same_string(s1, s2):
    # tưởng được đổi chỗ trong s1: chỉ cần s1, s2 có cùng số 'x'
    if s1.count("x") != s2.count("x"):
        return -1
    diff = sum(a != b for a, b in zip(s1, s2))
    return diff // 2


print("sai :", wrong_same_string("xx", "yy"), "| đúng:", Solution().minimumSwap("xx", "yy"))
print("sai :", wrong_same_string("xy", "yx"), "| đúng:", Solution().minimumSwap("xy", "yx"))

sai : -1 | đúng: 1
sai : 1 | đúng: 2


**Vì sao sai:** đề chỉ cho swap **giữa hai chuỗi**. Với `"xx"/"yy"` có thể giải bằng 1 swap chéo, nhưng code trên trả `-1`; với `"xy"/"yx"` code trả `1` (swap trong `s1`), không hợp lệ. **Sửa:** đọc kỹ đề; mô hình đúng là đếm `xy`/`yx`.

## 14. Mẹo & nhận dạng pattern

- Bài "số thao tác ít nhất" trên chuỗi nhị phân/hai loại ký tự → thường **chỉ cần đếm** vài loại vị trí, không cần mô phỏng.
- Luôn viết **BFS vét cạn** cho `n` nhỏ trước để kiểm chứng công thức tham lam.
- Tìm **bất biến chẵn lẻ** (invariant parity) để phát hiện trường hợp `-1`.
- Chứng minh tối ưu bằng **hàm thế** (potential function `Φ`): mỗi thao tác giảm `Φ` tối đa 1 → cận dưới; dựng được lời giải đạt `Φ` → cận trên.
- Bài cùng họ: 1888 Minimum Number of Flips to Make the Binary String Alternating, 1864 Minimum Number of Swaps to Make the Binary String Alternating, 2938 Separate Black and White Balls, 801 Minimum Swaps To Make Sequences Increasing.

## 15. Biến thể

1. **Bảng chữ cái lớn hơn** (`'x'`, `'y'`, `'z'`): có 6 loại lệch `(a, b)` với `a ≠ b`; hai vị trí cùng loại vẫn sửa bằng 1 swap, nhưng các vị trí khác loại tạo thành "chu trình" cần phân tích kỹ hơn — không còn công thức hai biến đơn giản.
2. **Chỉ được swap cùng chỉ số** (`s1[i] ↔ s2[i]`): swap như vậy chỉ biến `xy` thành `yx` (hoặc ngược lại), **không bao giờ** làm một vị trí lệch trở thành khớp → đáp án là `0` nếu đã bằng nhau, ngược lại `-1`.
3. **Trả về dãy swap cụ thể** → `build_swaps` ở Cách 3.
4. **Chi phí swap khác nhau** cho swap cùng chỉ số và khác chỉ số → cân nhắc giữa `2 · cost` và các phương án khác.
5. **Đếm số dãy swap tối ưu** → tổ hợp trên cách ghép cặp.

Demo biến thể 2:

In [24]:
def min_swap_same_index(s1, s2):
    """Chỉ được swap s1[i] <-> s2[i]: swap như vậy đổi 'xy' thành 'yx', không bao giờ làm khớp."""
    return 0 if s1 == s2 else -1


def brute_same_index(s1, s2):
    n = len(s1)

    def nb(st):
        a, b = st
        out = []
        for i in range(n):
            la, lb = list(a), list(b)
            la[i], lb[i] = lb[i], la[i]
            out.append(("".join(la), "".join(lb)))
        return out

    return bfs_min_steps((s1, s2), lambda st: st[0] == st[1], nb)


for n in range(1, 4):
    for t1 in itertools.product("xy", repeat=n):
        for t2 in itertools.product("xy", repeat=n):
            a, b = "".join(t1), "".join(t2)
            assert min_swap_same_index(a, b) == brute_same_index(a, b)
print("biến thể 2 khớp BFS")

biến thể 2 khớp BFS


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS đầy đủ | Duyệt mọi trạng thái chuỗi | hàm mũ | hàm mũ | Kiểm chứng với `n ≤ 5` |
| 2. Đếm + công thức | `xy//2 + yx//2 + 2·(xy%2)` | `O(n)` | `O(1)` | Code nộp |
| 3. Dựng dãy swap | Ghép cặp, áp dụng thật | `O(n)` | `O(n)` | Cần bằng chứng / in lời giải |
| 4. `Counter` + parity | `(xy+1)//2 + (yx+1)//2` | `O(n)` | `O(1)` | Viết ngắn nhất |
| 5. BFS `(xy, yx)` | BFS trên trạng thái thu gọn | `O(n)` | `O(n)` | Minh hoạ giảm chiều trạng thái |

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** `s1 = "xxxyy"`, `s2 = "yyyxx"`?

**Bài 2 (tính tay):** `s1 = "xyxyxy"`, `s2 = "yxyxyx"`: tính `xy`, `yx` và đáp án.

**Bài 3 (điền TODO):** viết `count_types(s1, s2)` trả về `(xy, yx)`.

**Bài 4 (tự viết lại):** viết `min_swap(s1, s2)` dùng `count_types` và công thức gọn `(xy+1)//2 + (yx+1)//2`.

**Bài 5 (mini challenge):** viết `min_swap_list(s1, s2)` trả về **danh sách swap** tối ưu (hoặc `None`), và tự kiểm chứng bằng `apply_swaps`.

In [25]:
def count_types(s1, s2):
    # TODO: trả về (xy, yx)
    raise NotImplementedError


def min_swap(s1, s2):
    # TODO
    raise NotImplementedError


def min_swap_list(s1, s2):
    # TODO: danh sách (i, j) nghĩa là swap s1[i] <-> s2[j]
    raise NotImplementedError


# Test gợi ý:
# assert count_types("xy", "yx") == (1, 1)
# assert min_swap("xx", "yy") == 1 and min_swap("xx", "xy") == -1
# sw = min_swap_list("xy", "yx"); a, b = apply_swaps("xy", "yx", sw); assert a == b and len(sw) == 2

## 18. Gợi ý 3 mức

**Bài 1–2:**
- Mức 1: đếm từng loại vị trí.
- Mức 2: Bài 1: `xy = 3`, `yx = 2` → tổng 5.
- Mức 3: Bài 2: `xy = 3`, `yx = 3` → `1 + 1 + 2`.

**Bài 3–4:**
- Mức 1: duyệt `zip(s1, s2)`.
- Mức 2: chỉ xét `a != b`; nếu `a == 'x'` là `xy`.
- Mức 3: kiểm tra parity trước khi áp công thức.

**Bài 5:**
- Mức 1: lưu chỉ số các vị trí `xy` và `yx`.
- Mức 2: ghép cặp cùng loại `(p, q)` → swap `(p, q)`.
- Mức 3: dư một cặp khác loại `p` (xy), `q` (yx) → `(p, p)` rồi `(p, q)`.

## 19. Đáp án tham khảo

**Bài 1:** `-1` (tổng lệch `5` lẻ). **Bài 2:** `xy = 3`, `yx = 3` → `4`.

In [26]:
assert Solution().minimumSwap("xxxyy", "yyyxx") == -1
assert Solution().minimumSwap("xyxyxy", "yxyxyx") == 4
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [27]:
def count_types(s1, s2):
    xy = sum(a == "x" and b == "y" for a, b in zip(s1, s2))
    yx = sum(a == "y" and b == "x" for a, b in zip(s1, s2))
    return xy, yx


def min_swap(s1, s2):
    xy, yx = count_types(s1, s2)
    if (xy + yx) % 2:
        return -1
    return (xy + 1) // 2 + (yx + 1) // 2


def min_swap_list(s1, s2):
    return build_swaps(s1, s2)


assert count_types("xy", "yx") == (1, 1)
assert min_swap("xx", "yy") == 1 and min_swap("xx", "xy") == -1
rng = random.Random(9)
for _ in range(300):
    n = rng.randint(1, 30)
    s1 = "".join(rng.choice("xy") for _ in range(n))
    s2 = "".join(rng.choice("xy") for _ in range(n))
    want = Solution().minimumSwap(s1, s2)
    assert min_swap(s1, s2) == want
    sw = min_swap_list(s1, s2)
    if want == -1:
        assert sw is None
    else:
        a, b = apply_swaps(s1, s2, sw)
        assert a == b and len(sw) == want
print("Bài 3, 4, 5 OK")

Bài 3, 4, 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template:**

```python
class Solution:
    def minimumSwap(self, s1: str, s2: str) -> int:
        xy = yx = 0
        for a, b in zip(s1, s2):
            if a != b:
                if a == "x": xy += 1
                else: yx += 1
        if (xy + yx) % 2:
            return -1
        return xy // 2 + yx // 2 + 2 * (xy % 2)
```

Ghi nhớ: **cùng loại → 1 swap / cặp; khác loại → 2 swap / cặp; tổng lẻ → -1**.

**Checklist trước khi submit:**

- [ ] Đếm **riêng** `xy` và `yx`, bỏ qua vị trí khớp.
- [ ] Điều kiện vô nghiệm là **tổng** `xy + yx` lẻ (không phải từng cái lẻ).
- [ ] Nhớ cộng `2` khi `xy`, `yx` cùng lẻ.
- [ ] Chỉ swap giữa hai chuỗi khác nhau (đúng đề).
- [ ] Đã đối chiếu với BFS vét cạn trên `n` nhỏ.